In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Read all the pdfs
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf_files/",
    glob="**/*.pdf",
    loader_cls= PyPDFLoader,
    loader_kwargs={},
    show_progress=False
)

documents = dir_loader.load()
# documents

In [3]:

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 8 PDF files to process

Processing: 1-s2.0-S221501612500264X-main.pdf
  ✓ Loaded 15 pages

Processing: agronomy-10-00573.pdf
  ✓ Loaded 20 pages

Processing: AS.pdf
  ✓ Loaded 1 pages

Processing: fsoil-5-1652058.pdf
  ✓ Loaded 27 pages

Processing: land-11-01027-v2.pdf
  ✓ Loaded 18 pages

Processing: PIIS2405844024011435.pdf
  ✓ Loaded 13 pages

Processing: Soil Health Analysis and Crop Suitability Prediction Using ML.pdf
  ✓ Loaded 6 pages

Processing: sustainability-12-09653-v2.pdf
  ✓ Loaded 25 pages

Total documents loaded: 125


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2025-06-07T18:29:09+05:30', 'crossmarkdomainexclusive': 'true', 'crossmarkdomains[2]': 'elsevier.com', 'author': 'Kapil Netaji Vhatkar', 'elsevierwebpdfspecifications': '7.0', 'subject': 'MethodsX, 14 (2025) 103418. doi:10.1016/j.mex.2025.103418', 'crossmarkmajorversiondate': '2010-04-23', 'crossmarkdomains[1]': 'sciencedirect.com', 'robots': 'noindex', 'moddate': '2025-06-07T18:30:15+05:30', 'authoritativedomain[1]': 'sciencedirect.com', 'keywords': '"Crop yield prediction"; "Sustainable agriculture"; "Soil health assessment"; "Iterative partitioning-ensemble filter"; "Back-propagation"; "Neural network"; "Multi-source data-fusion"; "Geographical information systems"', 'doi': '10.1016/j.mex.2025.103418', 'title': 'Enhancing prediction of crop yield and soil health assessment for sustainable agriculture using machine learning approach', 'authoritativedomain[2]': 'elsevier.com',

In [5]:
# Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 125 documents into 566 chunks

Example chunk:
Content: MethodsX 14 (2025) 103418 
Contents lists available at ScienceDirect 
MethodsX 
journal homepage: www.elsevier.com/locate/methodsx 
Enhancing prediction of crop yield and soil health assessment for 
s...
Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2025-06-07T18:29:09+05:30', 'crossmarkdomainexclusive': 'true', 'crossmarkdomains[2]': 'elsevier.com', 'author': 'Kapil Netaji Vhatkar', 'elsevierwebpdfspecifications': '7.0', 'subject': 'MethodsX, 14 (2025) 103418. doi:10.1016/j.mex.2025.103418', 'crossmarkmajorversiondate': '2010-04-23', 'crossmarkdomains[1]': 'sciencedirect.com', 'robots': 'noindex', 'moddate': '2025-06-07T18:30:15+05:30', 'authoritativedomain[1]': 'sciencedirect.com', 'keywords': '"Crop yield prediction"; "Sustainable agriculture"; "Soil health assessment"; "Iterative partitioning-ensemble filter"; "Back-propagation"; "Neural network"; "Multi-source data-f

[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2025-06-07T18:29:09+05:30', 'crossmarkdomainexclusive': 'true', 'crossmarkdomains[2]': 'elsevier.com', 'author': 'Kapil Netaji Vhatkar', 'elsevierwebpdfspecifications': '7.0', 'subject': 'MethodsX, 14 (2025) 103418. doi:10.1016/j.mex.2025.103418', 'crossmarkmajorversiondate': '2010-04-23', 'crossmarkdomains[1]': 'sciencedirect.com', 'robots': 'noindex', 'moddate': '2025-06-07T18:30:15+05:30', 'authoritativedomain[1]': 'sciencedirect.com', 'keywords': '"Crop yield prediction"; "Sustainable agriculture"; "Soil health assessment"; "Iterative partitioning-ensemble filter"; "Back-propagation"; "Neural network"; "Multi-source data-fusion"; "Geographical information systems"', 'doi': '10.1016/j.mex.2025.103418', 'title': 'Enhancing prediction of crop yield and soil health assessment for sustainable agriculture using machine learning approach', 'authoritativedomain[2]': 'elsevier.com',

### embedding And vectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
def __repr__(self):
    return f"EmbeddingManager(model_name='{self.model_name}', loaded={self.model is not None})"


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6077.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [9]:
texts = ["Hello world", "This is a test"]
embeddings = embedding_manager.generate_embeddings(texts)

print(embeddings.shape)

Generating embeddings for 2 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.61it/s]

Generated embeddings with shape: (2, 384)
(2, 384)


#### VectorStore

In [10]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [11]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2025-06-07T18:29:09+05:30', 'crossmarkdomainexclusive': 'true', 'crossmarkdomains[2]': 'elsevier.com', 'author': 'Kapil Netaji Vhatkar', 'elsevierwebpdfspecifications': '7.0', 'subject': 'MethodsX, 14 (2025) 103418. doi:10.1016/j.mex.2025.103418', 'crossmarkmajorversiondate': '2010-04-23', 'crossmarkdomains[1]': 'sciencedirect.com', 'robots': 'noindex', 'moddate': '2025-06-07T18:30:15+05:30', 'authoritativedomain[1]': 'sciencedirect.com', 'keywords': '"Crop yield prediction"; "Sustainable agriculture"; "Soil health assessment"; "Iterative partitioning-ensemble filter"; "Back-propagation"; "Neural network"; "Multi-source data-fusion"; "Geographical information systems"', 'doi': '10.1016/j.mex.2025.103418', 'title': 'Enhancing prediction of crop yield and soil health assessment for sustainable agriculture using machine learning approach', 'authoritativedomain[2]': 'elsevier.com',

In [12]:
# Convert the texts to embeddings
texts = [doc.page_content for doc in chunks]

# Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 566 texts...


Batches: 100%|██████████| 18/18 [00:15<00:00,  1.17it/s]

Generated embeddings with shape: (566, 384)


In [13]:
# Store in the vector database
vectorstore.add_documents(chunks,embeddings)

Adding 566 documents to vector store...
Successfully added 566 documents to vector store
Total documents in collection: 566


In [14]:
embeddings

array([[-0.06100946, -0.0409143 , -0.03139387, ..., -0.03388174,
        -0.07271801,  0.02931779],
       [-0.06212834, -0.05409767, -0.01931448, ..., -0.02957357,
        -0.0511999 ,  0.01244732],
       [-0.01511501, -0.0461655 ,  0.02050524, ..., -0.03938092,
        -0.06896764,  0.00025134],
       ...,
       [-0.00542818, -0.00467021,  0.04865432, ..., -0.06127703,
        -0.13393308, -0.02690919],
       [-0.01360692, -0.02495869,  0.04223167, ..., -0.00397643,
        -0.13976493, -0.09513368],
       [-0.01475711,  0.0611001 ,  0.06900168, ..., -0.03299574,
        -0.06387705, -0.1239273 ]], shape=(566, 384), dtype=float32)

### Retriever Pipeline from VectorDB

In [15]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [16]:
rag_retriever

In [17]:
rag_retriever.retrieve("What is soil health?")

Retrieving documents for query: 'What is soil health?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 76.08it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_684f2e02_532',
  'content': 'Sustainability 2020, 12, 9653 20 of 25\npractices that have caused many e ﬀects that reduce soil fertility, as land degradation factors are\nactivated in some regions, which aﬀects the availability of nutrients on one hand and increases carbon\nrelease to the atmosphere on the other [65–68]. The chemical properties of the soil are related to many\nsurrounding factors such as soil salinity, temperature, evapotranspiration and soil moisture, and other\nfactors that aﬀect the chemical balance in the soil, which is later reﬂected in the fertile state of the\nsoil [64–67]. The sand mapping unit is characterized by low quality, which is a result of the coarse soil\ntexture, saturated hydraulic conductivity and poor drainage conditions, where the physical properties\nof the soil depend on soil tillage conditions, organic additives, land use, fertilization and irrigation, in\naddition to soil conservation against the potential for degradation; this is 

In [18]:
rag_retriever.retrieve("Does Abhijit Santra study in BRAINWARE UNIVERSITY?")

Retrieving documents for query: 'Does Abhijit Santra study in BRAINWARE UNIVERSITY?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 93.42it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_1facb99a_177',
  'content': 'ABHIJIT SANTRA \nAspiring Software Developer \nContact no: +91- | Email:                   \nLinkedIn: Abhijit Santra | GitHub: abhijitsantra01             \nSabang, West Midnapore, West Bengal, India, 721467 \n \nCareer Objective:                Skills:      \n \n \n \n \n  \nEducation:         \nBRAINWARE UNIVERSITY  \nB. Tech, Computer Science and Engineering (AIML)  Projects:     \n(Anticipated – July, 2026)  \nCGPA – 8.63         \nMalpar Vivekananda Sikshaniketan, (WBCHSE) \nHigher Secondary Examination (2022) \nMarks - 94% \nChandkuri Union High School, (WBBSE) \nSecondary Examination (2020) \nMarks - 85% \n \nExperience:        \n \n \nCertifications:       \n \n \n \nHobby & Interest:       \n \n \n \n \n \n \n \nComputer Science (AI & ML) student with hands -on \nexperience in machine learning and NLP -based projects. \nFamiliar with Python, C++, and modern AI tools. Interested in \nbuilding practical, real -world solutions while cont

In [20]:
rag_retriever.retrieve("Explain Data Collection and Soil Sample Analysis",)

Retrieving documents for query: 'Explain Data Collection and Soil Sample Analysis'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.55it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_b5b9366f_297',
  'content': 'linked to different work ﬂows: Farmers provide soil data\n(NPK levels, moisture, pH values) that is processed and\nstored. Field Of ﬁcers can Access the same data to make\nrecommendations and provide advice to farmers.\n3. Data Flow: After login, farmers input the measured values\nwhich are stored in a farmers ’ database. The ﬁeld of ﬁcers\naccess these values, analyze them, and provide tailored\nrecommendations. Based on the soil data, farmers receive\nFIGURE 16\n(A) Soil Sample- 2A. (B) Soil Sample-3A. (C) Soil Sample-5A. (D) Soil Sample-8A.\nGunasekaran et al. 10.3389/fsoil.2025.1652058\nFrontiers in Soil Science frontiersin.org23',
  'metadata': {'total_pages': 27,
   'page': 22,
   'content_length': 630,
   'creator': 'PyPDF',
   'source': '..\\data\\pdf_files\\fsoil-5-1652058.pdf',
   'file_type': 'pdf',
   'keywords': 'soil fertility; machine learning; deep learning; random forest; decision tree; support vector machine; soil nutrients; c

### RAG Pipeline - VectorDB to LLM Output Generation

In [34]:
import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("OPENAI_API_KEY"))

True

In [39]:
import os
from typing import Optional
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

class GeminiLLM:
    def __init__(self, model_name: str = "gemini-2.5-flash", api_key: Optional[str] = None):
        """
        Initialize Google Gemini LLM
        
        Args:
            model_name: Gemini model name (e.g., gemini-1.5-flash, gemini-1.5-pro)
            api_key: Google API key (or set GOOGLE_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GOOGLE_API_KEY")
        
        if not self.api_key:
            raise ValueError("Google API key is required. Set GOOGLE_API_KEY environment variable or pass api_key parameter.")
        
        # Initialize the Gemini model via LangChain
        self.llm = ChatGoogleGenerativeAI(
            google_api_key=self.api_key,
            model=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Gemini LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}\n\nQuestion: {query}\n\nAnswer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [27]:
rag_retriever.retrieve("What RNDVI study says?")

Retrieving documents for query: 'What RNDVI study says?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.49it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_2dfac0d5_518',
  'content': 'of the total area, which was located in sand sheet units.\nTable 8. RNDVI classes in the study area.\nRNDVI Area (km2) Area (%)\nHigh 959 37.71\nModerate 268 10.54\nLow 1022 40.19\nVery low 294 11.56',
  'metadata': {'moddate': '2020-11-20T11:25:25+08:00',
   'creationdate': '2020-11-20T11:18:20+08:00',
   'creator': 'LaTeX with hyperref package',
   'source_file': 'sustainability-12-09653-v2.pdf',
   'file_type': 'pdf',
   'subject': 'Today, the global food security is one of the most pressing issues for humanity, and, according to Food and Agriculture Organisation (FAO), the increasing demand for food is likely to grow by 70% until 2050. In this current condition and future scenario, the agricultural production is a critical factor for global food security and for facing the food security challenge, with specific reference to many African countries, where a large quantities of rice are imported from other continents. According to FAO, to face

### Integration Vectordb Context pipeline With LLM output

In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv()

### Initialize the Gemini LLM (Ensure GOOGLE_API_KEY is in your .env file)
google_api_key = os.getenv("GEMINI_API_KEY")

if not google_api_key:
    raise ValueError("Google API key is missing. Please set GOOGLE_API_KEY.")

llm = ChatGoogleGenerativeAI(
    google_api_key=google_api_key,
    model="gemini-2.5-flash",  # Swapped to a fast Gemini model
    temperature=0.1,
    max_tokens=1024
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using Gemini LLM
    prompt = f"""Use the following context to answer the question concisely.
        
Context:
{context}
        
Question: {query}
        
Answer:"""
    
    # Pass the prompt string directly to invoke
    response = llm.invoke(prompt)
    return response.content

In [52]:
answer=rag_simple("Explain Chemical and Physical Soil Capability Indicators?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Explain Chemical and Physical Soil Capability Indicators?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.46it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Chemical Soil Capability Indicators (CSCI) are dynamic indicators that vary over time as a result of land management. They are chosen based on their sensitivity to disturbance and ability to execute soil ecosystem functions, and include EC, pH, ESP, CaCO3, and CEC. Physical indicators include depth.
